# FILE 1/3 — Luồng B, Chặng 1: Fine-tune COCO → CanTho (seed = 1407)

Notebook nay la **buoc dau tien** cua lo trinh tuan tu B, chay lai voi random seed moi
la **1407** (khac seed 42 da chay truoc do), theo yeu cau cua thay nham kiem chung tinh
on dinh cua ket luan.

**Vi tri trong toan bo quy trinh:**
- **FILE 1 (notebook nay):** COCO → CanTho  → sinh ra model B-CanTho (seed 1407)
- FILE 2: B-CanTho → HCMC → sinh ra model B cuoi cung (seed 1407)
- FILE 3: train luồng A (COCO→HCMC) + danh gia ca A va B tren 3 tap test + bang so sanh

**Luu y quan trong:**
- Chi doi **seed HUAN LUYEN** (seed=1407). Tap du lieu train/val/test GIU NGUYEN cach chia
  co san (khong chia lai), de bao dam so sanh cong bang voi ket qua seed=42.
- Model sinh ra duoc luu voi hau to `_seed1407`, KHONG de len ket qua seed=42 cu.
- Dung dung cau hinh Bang 3.4: imgsz=640, batch=16, epochs=100, patience=20, seed=1407.

**CHAY FILE NAY TRUOC. Sau khi xong, moi chay FILE 2.**

## Buoc 1: Cai dat thu vien

In [ ]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

## Buoc 2: Ket noi Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Buoc 3: Khai bao duong dan va cau hinh

In [ ]:
import os

SEED = 1407   # <=== SEED MOI theo yeu cau cua thay

# ===== MODEL KHOI TAO =====
# Baseline COCO (giai doan 1) - diem xuat phat cho chang 1 cua lo trinh B
BASELINE_COCO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"

# ===== DATASET (goc tren Drive) =====
DRIVE_DIR_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515"

# ===== DATASET (copy ve local de chay nhanh, on dinh) =====
LOCAL_DIR_CANTHO = "/content/cantho_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_B_CANTHO = "finetune_cantho_seed1407"   # ten thu muc model B-CanTho (seed moi)

# ===== CAU HINH HUAN LUYEN (dung Bang 3.4) =====
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 20
LR0 = 0.001

print("Kiem tra su ton tai:")
print("  Baseline COCO       :", os.path.exists(BASELINE_COCO))
print("  Dataset CanTho (Drive):", os.path.exists(DRIVE_DIR_CANTHO))

## Buoc 4: Copy dataset CanTho ve o cung local

Copy ve local giup train nhanh va on dinh hon. Neu gap loi "Transport endpoint is not
connected" (loi ket noi Drive tam thoi), chi can chay lai dung cell nay.

In [ ]:
import shutil, time

if os.path.exists(LOCAL_DIR_CANTHO):
    shutil.rmtree(LOCAL_DIR_CANTHO)
t0 = time.time()
try:
    shutil.copytree(DRIVE_DIR_CANTHO, LOCAL_DIR_CANTHO)
    print(f"[OK] Da copy CanTho ve {LOCAL_DIR_CANTHO} ({time.time()-t0:.1f}s)")
except shutil.Error as e:
    print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy, kiem tra lai so luong sau.")

DATA_YAML_CANTHO_LOCAL = f"{LOCAL_DIR_CANTHO}/data.yaml"
print("Yaml CanTho (local) ton tai:", os.path.exists(DATA_YAML_CANTHO_LOCAL))

## Buoc 5: Huan luyen chang 1 — COCO → CanTho (seed 1407)

Khoi tao tu Baseline COCO, fine-tune tren CanTho voi seed moi. Buoc nay co huan luyen
that nen mat thoi gian dang ke (tuy toc do GPU, khoang 1 gio). Nho bat GPU:
Runtime → Change runtime type → T4 GPU.

In [ ]:
from ultralytics import YOLO

model_b_cantho = YOLO(BASELINE_COCO)   # khoi tao tu Baseline COCO

results = model_b_cantho.train(
    data=DATA_YAML_CANTHO_LOCAL,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR0,
    seed=SEED,
    project=PROJECT_DIR,
    name=NAME_B_CANTHO,
    exist_ok=True,
    verbose=True,
)
print("\nDa huan luyen xong chang 1 (COCO -> CanTho, seed 1407).")
BEST_B_CANTHO = os.path.join(PROJECT_DIR, NAME_B_CANTHO, "weights", "best.pt")
print("Model B-CanTho luu tai:", BEST_B_CANTHO)
print("Ton tai:", os.path.exists(BEST_B_CANTHO))

## Buoc 6 (tuy chon): Danh gia nhanh chang 1 tren test CanTho

Buoc nay chi de kiem tra chang 1 da hoc tot chua truoc khi sang FILE 2. Khong bat buoc,
nhung nen chay de xac nhan model B-CanTho on dinh.

In [ ]:
BEST_B_CANTHO = os.path.join(PROJECT_DIR, NAME_B_CANTHO, "weights", "best.pt")
m = YOLO(BEST_B_CANTHO).val(
    data=DATA_YAML_CANTHO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_B_CANTHO + "_eval_cantho", exist_ok=True,
)
print("===== CHANG 1 (COCO->CanTho seed 1407) tren TEST CanTho =====")
print(f"mAP@0.5      : {float(m.box.map50):.4f}")
print(f"mAP@0.5:0.95 : {float(m.box.map):.4f}")
print(f"Precision    : {float(m.box.mp):.4f}")
print(f"Recall       : {float(m.box.mr):.4f}")
print("\n>>> DA XONG FILE 1. Tiep theo chay FILE 2 (B-CanTho -> HCMC).")